In [1]:
import os
from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

load_dotenv(dotenv_path="/Users/revathy_santhosh/rag-project/.env")

Settings.llm = Groq(model="openai/gpt-oss-20b", api_key=os.getenv("GROQ_API_KEY"))
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("LLM and embedding model configured.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LLM and embedding model configured.


In [2]:
import glob
import fitz  # this is PyMuPDF's import name
from llama_index.core import Document

documents = []
for pdf_path in glob.glob("/Users/revathy_santhosh/rag-project/data/*.pdf"):
    pdf = fitz.open(pdf_path)
    file_name = pdf_path.split("/")[-1]
    for page_num, page in enumerate(pdf, start=1):
        text = page.get_text()
        if text.strip():  # skip blank pages
            documents.append(Document(text=text, metadata={"file_name": file_name, "page": page_num}))

print(f"Loaded {len(documents)} page-documents from {len(set(d.metadata['file_name'] for d in documents))} files")
for fname in sorted(set(d.metadata["file_name"] for d in documents)):
    total_chars = sum(len(d.text) for d in documents if d.metadata["file_name"] == fname)
    print(f"- {fname}: {total_chars} chars")

Loaded 135 page-documents from 5 files
- AI_Engineering_RAG_chapter.pdf: 30755 chars
- BDes_AdmissionsHandbook2026-27.pdf: 77802 chars
- Edward_Guiliano_Global_Fellowship_23-24_Annual_Report.pdf: 23441 chars
- HBR_How_Apple_Is_Organized_For_Innovation-4.pdf: 36606 chars
- SR_InternationalStudentSupport_ComingToNewYork_English_2023.pdf: 82515 chars


In [3]:
for doc in documents:
    if "Edward_Guiliano" in doc.metadata.get("file_name", ""):
        print(doc.text[:2000])
        break
    

ANNUAL REPORT
2023/2024
Prepared By :
Askia VanOmmeren
Assistant Director of Experiential Education



In [4]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

chroma_client = chromadb.PersistentClient(path="/Users/revathy_santhosh/rag-project/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("rag_project")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)
print("Index built and stored in Chroma.")

Index built and stored in Chroma.


In [5]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("What does the HBR article say about how Apple organizes for innovation?")
print(response)

print("\n--- Sources used ---")
for node in response.source_nodes:
    print(f"- {node.metadata.get('file_name')} (page {node.metadata.get('page')}) — score: {node.score:.3f}")

The article explains that Apple’s approach to innovation is built around a structure in which experts lead other experts, creating a culture where deep technical and design expertise drives decision‑making and product development.

--- Sources used ---
- HBR_How_Apple_Is_Organized_For_Innovation-4.pdf (page 1) — score: 0.779
- HBR_How_Apple_Is_Organized_For_Innovation-4.pdf (page 1) — score: 0.779
- HBR_How_Apple_Is_Organized_For_Innovation-4.pdf (page 1) — score: 0.779


In [6]:
eval_questions = [
    {"question": "According to the HBR article, how is Apple organized to support innovation?",
     "ground_truth": "Apple is organized around functional expertise rather than by product lines or business units, with senior leaders who are deep experts in their discipline, enabling fast, high-quality decisions at scale."},
    {"question": "What is the last date for submitting the B.Des application form for NID's 2026-27 admissions?",
     "ground_truth": "11:59 pm (midnight) on Monday, 01 December 2025."},
    {"question": "When is the Design Aptitude Test (DAT) Prelims scheduled for NID's B.Des 2026-27 admissions?",
     "ground_truth": "Sunday, 21 December 2025."},
    {"question": "What NYIT office publishes the 'Coming to New York' guide for international students?",
     "ground_truth": "NYIT's Office of International Education."},
    {"question": "What visa statuses does NYIT's international student guide cover for obtaining student status?",
     "ground_truth": "F-1 or J-1 student status."},
    {"question": "Who prepared the Edward Guiliano Global Fellowship 2023/2024 Annual Report?",
     "ground_truth": "Askia VanOmmeren, Assistant Director of Experiential Education."},
    {"question": "According to the AI Engineering book's RAG chapter, what are the main steps in a RAG system's workflow?",
     "ground_truth": "A RAG system loads documents, chunks them, embeds the chunks, retrieves the most relevant chunks for a given query, and generates an answer using that retrieved context."},
]
eval_data = []

for item in eval_questions:
    response = query_engine.query(item["question"])
    eval_data.append({
        "user_input": item["question"],
        "response": str(response),
        "retrieved_contexts": [node.text for node in response.source_nodes],
        "reference": item["ground_truth"],
    })
    print(f"done: {item['question'][:60]}...")

print(f"\nCollected {len(eval_data)} Q&A pairs.")

#eval_data = {"user_input": [], "response": [], "retrieved_contexts": [], "reference": []}

#for item in eval_questions:
    #response = query_engine.query(item["question"])
    #eval_data["user_input"].append(item["question"])
    #eval_data["response"].append(str(response))
    #eval_data["retrieved_contexts"].append([node.text for node in response.source_nodes])
    #eval_data["reference"].append(item["ground_truth"])
    #print(f"done: {item['question'][:60]}...")

#print(f"\nCollected {len(eval_data['user_input'])} Q&A pairs.")

done: According to the HBR article, how is Apple organized to supp...
done: What is the last date for submitting the B.Des application f...
done: When is the Design Aptitude Test (DAT) Prelims scheduled for...
done: What NYIT office publishes the 'Coming to New York' guide fo...
done: What visa statuses does NYIT's international student guide c...
done: Who prepared the Edward Guiliano Global Fellowship 2023/2024...
done: According to the AI Engineering book's RAG chapter, what are...

Collected 7 Q&A pairs.


In [8]:
from ragas.run_config import RunConfig

my_run_config = RunConfig(timeout=180, max_retries=10, max_wait=90, max_workers=1)

result = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(llm=ragas_llm),
        AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
        ContextPrecision(llm=ragas_llm),
        ContextRecall(llm=ragas_llm),
    ],
    run_config=my_run_config,
    show_progress=True,
)

print(result)
df = result.to_pandas()
df.to_csv("ragas_eval_results.csv", index=False)
df

#from ragas import evaluate, EvaluationDataset
#from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
#from ragas.llms import LlamaIndexLLMWrapper
#from ragas.embeddings import LlamaIndexEmbeddingsWrapper

#ragas_llm = LlamaIndexLLMWrapper(Settings.llm)
#ragas_embeddings = LlamaIndexEmbeddingsWrapper(Settings.embed_model)

#dataset = EvaluationDataset.from_dict(eval_data)

#result = evaluate(
#    dataset=dataset,
#    metrics=[
#        Faithfulness(llm=ragas_llm),
#        AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
#        ContextPrecision(llm=ragas_llm),
#       ContextRecall(llm=ragas_llm),
#    ],
#)

#print(result)
#df = result.to_pandas()
#df

Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

Retrying llama_index.llms.openai.base.OpenAI._achat in 4 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m4s286v7e2sb3dvwjkxq31z3` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7465, Requested 979. Please try again in 3.33s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}.
Retrying llama_index.llms.openai.base.OpenAI._achat in 2 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m4s286v7e2sb3dvwjkxq31z3` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6789, Requested 1394. Please try again in 1.3725s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}.


{'faithfulness': 0.7143, 'answer_relevancy': 0.7667, 'context_precision': 0.7857, 'context_recall': 0.7143}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,"According to the HBR article, how is Apple org...",[PHOTOGRAPHER MIKAEL JANSSON\nHow Apple Is \nO...,Apple is organized around a structure where ex...,Apple is organized around functional expertise...,1.0,0.807043,0.0,0.0
1,What is the last date for submitting the B.Des...,[1 \nNATIONAL INSTITUTE OF DESIGN: B.DES ADMIS...,The final deadline for submitting the B.Des ap...,"11:59 pm (midnight) on Monday, 01 December 2025.",1.0,0.846517,1.0,1.0
2,When is the Design Aptitude Test (DAT) Prelims...,[25 \nNATIONAL INSTITUTE OF DESIGN: B.DES ADMI...,The DAT Prelims for NID’s B.Des 2026‑27 admiss...,"Sunday, 21 December 2025.",1.0,0.918846,1.0,1.0
3,What NYIT office publishes the 'Coming to New ...,[3 \n \nWelcome To NYIT! \nStudying in a forei...,The guide is published by NYIT’s Office of Int...,NYIT's Office of International Education.,0.0,0.694258,1.0,1.0
4,What visa statuses does NYIT's international s...,[3 \n \nWelcome To NYIT! \nStudying in a forei...,The guide covers the F‑1 and J‑1 student visa ...,F-1 or J-1 student status.,1.0,0.870947,1.0,1.0
5,Who prepared the Edward Guiliano Global Fellow...,[ANNUAL REPORT\n2023/2024\nPrepared By :\nAski...,Askia VanOmmeren.,"Askia VanOmmeren, Assistant Director of Experi...",0.0,0.454783,1.0,1.0
6,According to the AI Engineering book's RAG cha...,"[DailyDoseofDS.com \n \nConsequently, the LLM ...",The excerpt you shared does not list the speci...,"A RAG system loads documents, chunks them, emb...",1.0,0.774195,0.5,0.0
